# Lab F5. Forecasting with many predictors

Thirty indicators and two hundred quarters: least squares on all of them fits the past well and forecasts badly.
This lab compares the standard remedies: summarise the indicators by a few factors, shrink the coefficients, or
average pretested models over bootstrap samples.

**Help:** for any pymetrix function, type its name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call. The last cell lists every pymetrix function, class and method this lab uses.

Topics: principal components and the Bai-Ng criteria for the number of factors; diffusion-index forecasts; ridge
and lasso with the penalty chosen by time-series cross-validation; bagging (Elliott and Timmermann, chapters 5, 6,
10 and 11).

The data: two unobserved AR(1) factors drive 30 indicators, $x_{i,t} = \lambda_i' f_t + e_{i,t}$, and the target,
$y_t = 0.8\,f_{1,t-1} + 0.4\,f_{2,t-1} + \varepsilon_t$.

In [ ]:
import numpy as np
import pandas as pd

from pymetrix import PCA, Bagging, Dataset, Equation, backtest

rng = np.random.default_rng(10)
T, N = 200, 30
f = np.zeros((T, 2))
for t in range(1, T):
    f[t] = [0.8, 0.5] * f[t - 1] + rng.standard_normal(2)
loadings = rng.standard_normal((N, 2))
x = f @ loadings.T + rng.standard_normal((T, N))
y = np.r_[0.0, 0.8 * f[:-1, 0] + 0.4 * f[:-1, 1]] + rng.standard_normal(T)
names = [f"x{i + 1}" for i in range(N)]
frame = pd.DataFrame(np.column_stack([y, x]), columns=["y", *names], index=pd.period_range("1975Q1", periods=T, freq="Q"))
data = Dataset.from_frame(frame, source="simulated two-factor panel")

## 1. How many factors?

Principal components of the standardized indicators. Bai and Ng's criteria trade the fit of $k$ components
against a penalty that grows with $k$ and shrinks with both $N$ and $T$; each picks the $k$ with the smallest
value.

In [ ]:
pca = PCA(names).fit(data)
pca.factor_criteria()

**Check it yourself.** Factors are identified only up to a rotation, so compare spaces, not series: regress each
true factor on the first two components. An $R^2$ near one means the components have found it.

In [ ]:
import statsmodels.api as sm

pcs = pca.scores[["pc1", "pc2"]].to_numpy()
r2 = [sm.OLS(f[:, j], sm.add_constant(pcs)).fit().rsquared for j in range(2)]
{"R2, factor 1 on two components": round(float(r2[0]), 3), "R2, factor 2": round(float(r2[1]), 3), "share of variance, first two": round(float(pca.explained.iloc[:2].sum()), 3)}

## 2. Five forecasts of $y$, one quarter ahead

- the historical mean;
- "kitchen sink" least squares on all 30 lagged indicators;
- a diffusion index: least squares on the first two components, lagged;
- bagging the 30-indicator regression: keep the indicators with $|t| > 1.96$ in each bootstrap sample, and average;
- ridge and lasso, if scikit-learn is installed (`pip install pymetrix[ml]`).

Every forecast is made in real time from 2000Q1 to 2024Q3. For the diffusion index that means the components are
re-estimated at each origin from the indicators known then: a function `(data, origin) -> fit` does it, and a
backtest accepts any such function. Ridge and lasso choose their penalty by time-series cross-validation; doing
that at every origin (which a backtest of `Ridge()` does) takes a minute, so here each penalty is chosen once,
from the data before the first forecast, and then kept.

In [ ]:
everything = " + ".join(f"{name}[-1]" for name in names)
origins = ("2000Q1", "2024Q3")


class DiffusionIndex:
    """At one origin: the first two components of the indicators to then, and a regression of y on them, lagged."""

    target = "y"

    def __init__(self, data, origin):
        self.pca = PCA(names, n=2).fit(data, span=(None, str(origin)))
        self.equation = Equation("y ~ pc1[-1] + pc2[-1]").fit(self._with_factors(data), span=(None, str(origin)))

    def _with_factors(self, data):
        scores = self.pca.transform(data)
        return data.with_series(pc1=scores["pc1"], pc2=scores["pc2"])

    def forecast(self, data, span, dynamic=False):
        return self.equation.forecast(self._with_factors(data), span, dynamic=False)


bts = {
    "mean": backtest(Equation("y ~ 1").fit(data), data, origins, horizon=1, label="mean"),
    "kitchen sink": backtest(Equation(f"y ~ {everything}").fit(data), data, origins, horizon=1, label="kitchen sink"),
    "diffusion index": backtest(DiffusionIndex, data, origins, horizon=1, label="diffusion index"),
    "bagging": backtest(Equation(f"y ~ {everything}").fit(data, method=Bagging(reps=100)), data, origins, horizon=1, label="bagging"),
}
try:
    import warnings

    from pymetrix import Lasso, Ridge

    penalties = {}
    for name, kind in (("ridge", Ridge), ("lasso", Lasso)):
        with warnings.catch_warnings():  # the solver can stop short of its very tight tolerance at the smallest penalties
            warnings.simplefilter("ignore")
            chosen = Equation(f"y ~ {everything}").fit(data, span=(None, "1999Q4"), method=kind(folds=5))
        penalties[name] = chosen.stats["penalty"]
        bts[name] = backtest(Equation(f"y ~ {everything}").fit(data, method=kind(alpha=penalties[name])), data, origins, horizon=1, label=name)
except ImportError as exc:  # pragma: no cover - scikit-learn not installed
    print(f"ridge and lasso skipped: {exc}")
accuracy = pd.DataFrame({name: bt.scores(["y"]).iloc[0] for name, bt in bts.items()}).T[["RMSE", "MAE"]]
accuracy["RMSE relative to the mean"] = accuracy["RMSE"] / accuracy.loc["mean", "RMSE"]
accuracy

The best possible one-step forecast, from the true factors and coefficients, has RMSE 1 (the standard deviation
of $\varepsilon$).

**Check it yourself.** The diffusion-index forecast from the first origin, by hand: components from the
indicators to 2000Q1 (standardized with that sample's means and standard deviations), a least-squares fit of $y$
on them lagged, and the 2000Q2 forecast from the 2000Q1 components.

In [ ]:
past = frame.loc[: pd.Period("2000Q1", freq="Q")]
z = (past[names] - past[names].mean()) / past[names].std(ddof=1)
values, vectors = np.linalg.eigh(np.cov(z.to_numpy(), rowvar=False))
top = vectors[:, np.argsort(values)[::-1][:2]]
scores = z.to_numpy() @ top
X = sm.add_constant(scores[:-1])
b = np.linalg.lstsq(X, past["y"].to_numpy()[1:], rcond=None)[0]
by_hand = b @ np.r_[1.0, scores[-1]]  # signs of the components do not matter: the coefficients flip with them
first = bts["diffusion index"].errors.iloc[0]
assert np.isclose(by_hand, first["forecast"])
print(f"first diffusion-index forecast by hand {by_hand:.6f}; pymetrix {first['forecast']:.6f}")

**Questions.**

1. The kitchen sink fits the past best (compare the in-sample $R^2$ of the 30-indicator regression with the
   diffusion index's). Why does it forecast worst?
2. Which indicators does bagging keep most often (`Equation(f"y ~ {everything}").fit(data,
   method=Bagging()).summary()` lists them)? Do they load heavily on the first factor (`pca.loadings`)?
3. Test whether the diffusion index beats ridge (`bts["diffusion index"].compare(bts["ridge"], "y", 1)`). Would
   you expect it to when the data really have a factor structure? When would ridge or lasso win?
4. IC_p3 picks far more factors than IC_p1 and IC_p2. Look at its penalty: why is it the weakest when $N$ is
   small relative to $T$?

In the app: **Estimate → Principal components** (the Number of factors tab has the Bai-Ng criteria);
**Estimate → Equation** has Ridge, Lasso, Elastic net and Bagging among its methods; **Forecast → Backtest**
evaluates any of them.

## pymetrix used in this lab

Every pymetrix function, class and method this lab calls, with its arguments and the first line of its documentation. For the full documentation, type the name and a question mark in a cell (`backtest?`), or press Shift+Tab inside the brackets of a call.

### From `pymetrix`

| | |
|---|---|
| `backtest(fitted, data, origins, horizon, vintages, variable, label, window)` | Re-estimate at each origin from `origins[0]` to `origins[1]` and forecast `horizon` periods ahead. |
| `Bagging(critical, reps, block, seed)` | Bootstrap aggregation of a pretest (Inoue and Kilian 2008), averaging the coefficients over bootstrap samples. |
| `Equation(text, params, family, priors, name, errors, variance)` | An equation written as text: `"c ~ y + c[-1]"` to estimate, or `"y = c + i + g"` (an identity). |
| `Lasso(alpha, folds, backend_options)` | The lasso: least squares with a penalty on the sum of absolute coefficients, which sets some exactly to zero (selection). |
| `PCA(variables, n, standardize)` | Principal components of a set of series: the weighted averages that explain most of their variance. |
| `Ridge(alpha, folds, backend_options)` | Ridge regression: least squares with a penalty on the sum of squared coefficients, shrinking them all toward zero. |

### Methods and properties of the results

| | |
|---|---|
| `Backtest.scores(variables)` | n, RMSE, MAE and mean error by variable and horizon (a two-level index). |
| `Dataset.from_frame(df, time, entity, freq, variables, meta, source)` | Build a Dataset from a DataFrame, naming its time and entity columns. |
| `Dataset.with_series(within, **exprs)` | Add or replace series, from expressions or from aligned Series; `None` makes a series of missing values (EViews' `series y`). |
| `Equation.fit(data, span, where, method, weights, missing)` | Estimate the equation on `data`: by least squares unless `method` says otherwise (`OLS(cov="hac")`, `IV(...)`, `ML()`, `Ridge()`, ...). |
| `FittedEquation.forecast(data, span, dynamic, alpha)` | Forecasts of the left-hand side with standard errors and a (1 - alpha) interval. |
| `FittedPCA.explained` | The share of the total variance each component explains. |
| `FittedPCA.factor_criteria(kmax)` | Bai and Ng's (2002) information criteria for the number of factors: IC_p1, IC_p2 and IC_p3 for k = 1 to `kmax` (default min(8, N - 1)), with the k each picks (the smallest value). |
| `FittedPCA.scores` | The components in each period of the sample: one column per component (pc1, pc2, ...). |
| `FittedPCA.transform(data)` | The components for (other) data, with this fit's means, scales and loadings. |
| `PCA.fit(data, span)` | The components of the series over the periods of `span` (first, last) where none is missing. |